# 第 02 天：收益率与标签

> 来自《30 天因子研究计划》第 2 天  
> 主题：收益率与标签  
> 必做：收益率计算  
> 选做：不同持有期收益  
> 目标产出：构建未来 20 日收益标签

---

## 0. 今天你要真正学会什么？

第 1 天我们建立了因子投资框架：因子值要拿去解释或预测未来收益。

今天要解决一个非常具体、也非常容易出错的问题：

> 给定一张价格表，如何正确计算收益率，并为每个股票、每个交易日构建“未来 20 日收益标签”？

学完以后，你应该能回答这 6 个问题：

1. 简单收益率和对数收益率有什么区别？
2. 为什么因子研究通常要使用复权价格？
3. “未来 20 日收益标签”到底从哪一天算到哪一天？
4. 为什么 `shift(-20)` 既强大又危险？
5. 不同持有期收益如何批量构造？
6. 如何避免未来函数和日期错位？

一句话版：

> 标签不是随手算出来的收益率，而是和因子时间点严格对齐的未来收益。

---

## 1. 先建立直觉：标签是答案，因子是考生

想象你在做一场考试。

- 因子值：考生交上来的答案。
- 未来收益：标准答案。
- IC、分层回测：判卷方式。

如果标准答案拿错了，后面的评分就全废了。

在因子研究里，最常见的任务是：


在 t 日，我们看到每只股票的因子值。
然后观察 t 日之后一段时间的收益。
看因子值高低能不能解释未来收益高低。


所以一个最简洁的数据结构是：

| date | ticker | factor_value | future_20d_ret |
| --- | --- | ---: | ---: |
| 2024-01-02 | A | 1.23 | 4.5% |
| 2024-01-02 | B | -0.80 | -1.2% |
| 2024-01-03 | A | 1.10 | 3.8% |

这里最重要的是：


factor_value 是当时能知道的信息
future_20d_ret 是之后才发生的结果


研究时可以用未来收益做标签，因为你是在历史回测；但构造因子时绝不能偷看未来。

---

## 2. 三个核心词：收益率、持有期、标签

### 2.1 收益率：价格变化的标准化表达

如果一只股票从 10 元涨到 11 元，收益率是：


(11 / 10) - 1 = 10%


这比直接说“涨了 1 元”更有可比性。

因为：

- 10 元涨 1 元，是 10%。
- 100 元涨 1 元，是 1%。

收益率把价格变化放到了同一个尺度上。

### 2.2 持有期：你从买入到卖出等了多久

持有期可以是：

- 1 日：明天卖。
- 5 日：约一周后卖。
- 20 日：约一个月后卖。
- 60 日：约一个季度后卖。

不同因子的有效周期不同。

例子：

- 短期反转因子可能看 1-5 日。
- 动量因子可能看 20-120 日。
- 价值、质量因子可能看 20-250 日。

所以今天的选做内容“不同持有期收益”非常重要。

### 2.3 标签：给机器或统计检验看的未来答案

在机器学习里，标签常叫 `y`。

在因子研究里，标签通常是未来收益：


y_{t,i} = stock_i 从 t 到 t+20 的未来收益


如果我们今天构造的是未来 20 日收益标签，最常见写法是：


future_20d_ret(t) = price(t+20) / price(t) - 1


用 Pandas 写就是：


future_20d_ret = price.shift(-20) / price - 1


这行代码很短，但它背后的时间含义必须想清楚。

---

## 3. 准备 Python 环境

下面的代码可以放在 Jupyter Notebook、VS Code 或普通 Python 文件里运行。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260703)


如果你本地缺少包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 4. 从价格到收益率

我们先用一只股票做最小实验。

### 4.1 构造一条模拟价格曲线


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=120)

daily_noise = rng.normal(loc=0.0006, scale=0.018, size=len(dates))
close = pd.Series(
    100 * np.cumprod(1 + daily_noise),
    index=dates,
    name="close"
)

price_df = close.to_frame()
price_df.head()


这里我们构造了 120 个交易日的收盘价。

### 4.2 简单收益率

简单收益率公式：


simple_return(t) = price(t) / price(t-1) - 1


Pandas 写法：


In [ ]:
price_df["simple_ret"] = price_df["close"].pct_change()
price_df.head(8)


第一天没有上一日价格，所以收益率是缺失值。

这是合理的，不要强行填 0。填 0 会把“没有数据”伪装成“当天没涨没跌”。

### 4.3 对数收益率

对数收益率公式：


log_return(t) = log(price(t) / price(t-1))


Pandas 写法：


In [ ]:
price_df["log_ret"] = np.log(price_df["close"] / price_df["close"].shift(1))
price_df.head(8)


### 4.4 简单收益率 vs 对数收益率

简单收益率更符合直观，也更常用于组合收益计算。

对数收益率有一个漂亮性质：多期收益可以相加。


In [ ]:
example = price_df[["simple_ret", "log_ret"]].dropna().head(5)

simple_5d = (1 + example["simple_ret"]).prod() - 1
log_5d = example["log_ret"].sum()
log_5d_to_simple = np.exp(log_5d) - 1

print("5 日简单收益复利:", f"{simple_5d:.4%}")
print("5 日对数收益求和后转简单收益:", f"{log_5d_to_simple:.4%}")
example


你会看到两个结果非常接近，理论上它们是等价转换：


sum(log returns) = log(final_price / initial_price)


### 4.5 画图看看


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

price_df["close"].plot(ax=axes[0], title="模拟收盘价")
axes[0].set_ylabel("Price")

price_df["simple_ret"].plot(ax=axes[1], title="日简单收益率")
axes[1].set_ylabel("Return")

plt.tight_layout()
plt.show()


价格看起来有趋势，收益率看起来像围绕 0 上下波动的噪声。

因子研究大多研究的是收益率，而不是价格本身。

---

## 5. 为什么要用复权价格？

真实股票会发生：

- 分红
- 送股
- 拆股
- 配股
- 除权除息

如果你只看原始收盘价，可能会把分红除权造成的价格跳落误认为暴跌。

### 5.1 一个拆股例子

假设一只股票前一天 100 元，第二天 1 拆 2，价格变成 50 元。

你真的亏了 50% 吗？

不是。你持有的股数翻倍了，总资产没有因为拆股本身减少。

我们用代码模拟：


In [ ]:
split_dates = pd.bdate_range("2024-04-01", periods=8)

raw_close = pd.Series(
    [96, 98, 100, 50, 51, 52, 53, 54],
    index=split_dates,
    name="raw_close"
)

adj_close = pd.Series(
    [48, 49, 50, 50, 51, 52, 53, 54],
    index=split_dates,
    name="adj_close"
)

split_demo = pd.concat([raw_close, adj_close], axis=1)
split_demo["raw_ret"] = split_demo["raw_close"].pct_change()
split_demo["adj_ret"] = split_demo["adj_close"].pct_change()

split_demo


对比结果：

- 原始价格会显示大约 -50%。
- 复权价格会显示拆股当天收益约 0%。

### 5.2 结论

因子研究中，如果你要计算历史收益率，通常应该使用复权价格。

常见选择：

- 前复权价格：适合看历史图、做历史收益率计算。
- 后复权价格：适合某些长期净值复原场景。
- 不复权价格：适合研究当时市场实际成交价格，但不能直接拿来算长期收益。

初学阶段记住一句：

> 做收益率标签时，优先用复权收盘价，避免把除权除息误判成收益波动。

---

## 6. 构建未来 N 日收益标签

现在进入今天最重要的部分。

### 6.1 未来 1 日收益

如果今天是 `t`，明天是 `t+1`，未来 1 日收益是：


future_1d_ret(t) = price(t+1) / price(t) - 1


代码：


In [ ]:
price_df["future_1d_ret"] = price_df["close"].shift(-1) / price_df["close"] - 1
price_df[["close", "simple_ret", "future_1d_ret"]].head(8)


注意区别：

- `simple_ret` 是今天相对昨天的收益，是已经发生的。
- `future_1d_ret` 是明天相对今天的收益，是未来标签。

### 6.2 未来 20 日收益


In [ ]:
holding_days = 20
price_df["future_20d_ret"] = price_df["close"].shift(-holding_days) / price_df["close"] - 1

price_df[["close", "future_1d_ret", "future_20d_ret"]].head(25)


最后 20 行会是缺失值，因为没有足够的未来价格。

这是正确的。

不要为了“填满数据”把最后 20 行乱填。

### 6.3 直观验证某一天标签

我们手工检查第 10 个交易日的未来 20 日收益。


In [ ]:
check_pos = 10
entry_date = price_df.index[check_pos]
exit_date = price_df.index[check_pos + 20]

entry_price = price_df.loc[entry_date, "close"]
exit_price = price_df.loc[exit_date, "close"]
manual_ret = exit_price / entry_price - 1
auto_ret = price_df.loc[entry_date, "future_20d_ret"]

print("起始日期:", entry_date.date())
print("结束日期:", exit_date.date())
print("起始价格:", round(entry_price, 4))
print("结束价格:", round(exit_price, 4))
print("手工计算:", f"{manual_ret:.4%}")
print("代码标签:", f"{auto_ret:.4%}")


这个检查习惯很重要。

每次写标签函数，都要随便抽几行手工验证，尤其是 `shift` 的方向。

---

## 7. `shift` 的方向：初学者高发错误

### 7.1 `shift(1)` 是过去


In [ ]:
shift_demo = pd.DataFrame({"close": price_df["close"].head(6)})
shift_demo["close_shift_1"] = shift_demo["close"].shift(1)
shift_demo["close_shift_minus_1"] = shift_demo["close"].shift(-1)
shift_demo


含义：

- `shift(1)`：把昨天价格拿到今天这一行。
- `shift(-1)`：把明天价格拿到今天这一行。

所以：


In [ ]:
shift_demo["past_1d_ret"] = shift_demo["close"] / shift_demo["close"].shift(1) - 1
shift_demo["future_1d_ret"] = shift_demo["close"].shift(-1) / shift_demo["close"] - 1
shift_demo


### 7.2 最容易写反的地方

错误写法：


future_ret = price / price.shift(20) - 1


这其实是过去 20 日收益，不是未来 20 日收益。

正确写法：


future_ret = price.shift(-20) / price - 1


记忆方法：

> 未来价格要被搬到今天这一行，所以用负数 shift。

---

## 8. 不同持有期收益

第 2 天的选做内容是“不同持有期收益”。这一步会在后面判断因子衰减时反复使用。

### 8.1 写一个通用函数


In [ ]:
def forward_return(price: pd.Series, holding_days: int) -> pd.Series:
    """
    用收盘价构建未来 holding_days 个交易日的简单收益率。

    标签含义：
    在 t 日价格为 price[t]，
    未来 holding_days 个交易日后价格为 price[t + holding_days]，
    收益 = price[t + holding_days] / price[t] - 1。
    """
    return price.shift(-holding_days) / price - 1


for h in [1, 5, 10, 20, 60]:
    price_df[f"future_{h}d_ret"] = forward_return(price_df["close"], h)

price_df[["close", "future_1d_ret", "future_5d_ret", "future_20d_ret", "future_60d_ret"]].head(12)


### 8.2 比较不同持有期标签


In [ ]:
label_cols = ["future_1d_ret", "future_5d_ret", "future_10d_ret", "future_20d_ret", "future_60d_ret"]

summary = price_df[label_cols].agg(["count", "mean", "std", "min", "max"]).T
summary["mean"] = summary["mean"].map(lambda x: f"{x:.2%}")
summary["std"] = summary["std"].map(lambda x: f"{x:.2%}")
summary["min"] = summary["min"].map(lambda x: f"{x:.2%}")
summary["max"] = summary["max"].map(lambda x: f"{x:.2%}")
summary


你会看到：

- 持有期越长，可用样本越少。
- 持有期越长，收益波动通常越大。
- 不同持有期对应不同研究问题。

### 8.3 画出不同持有期标签


In [ ]:
price_df[label_cols].plot(title="不同持有期的未来收益标签")
plt.ylabel("Forward Return")
plt.show()


这张图有点“乱”，但它能提醒你：

> 标签不是静态答案，持有期一变，答案就变。

---

## 9. 多股票场景：真正的因子研究数据形态

真实研究不是只有一只股票，而是每天几百、几千只股票。

我们来构造一个小股票池。

### 9.1 构造多股票价格矩阵


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=180)
tickers = ["AAA", "BBB", "CCC", "DDD", "EEE"]

market_ret = rng.normal(0.0003, 0.010, size=len(dates))

price_wide = pd.DataFrame(index=dates)

for i, ticker in enumerate(tickers):
    beta = 0.7 + i * 0.25
    alpha = (i - 2) * 0.00005
    noise = rng.normal(0, 0.012 + i * 0.002, size=len(dates))
    ret = alpha + beta * market_ret + noise
    price_wide[ticker] = 50 * (1 + i * 0.2) * np.cumprod(1 + ret)

price_wide.head()


这种宽表很适合做价格计算：


index = date
columns = ticker
values = adjusted close price


### 9.2 批量计算日收益率


In [ ]:
daily_ret_wide = price_wide.pct_change()
daily_ret_wide.head()


### 9.3 批量计算未来 20 日收益标签


In [ ]:
future_20d_ret_wide = price_wide.shift(-20) / price_wide - 1
future_20d_ret_wide.head()


这就是最直接的未来 20 日标签矩阵。

### 9.4 从宽表转成长表

后面做 IC、分组回测、机器学习时，长表通常更方便：


date | ticker | future_20d_ret


代码：


In [ ]:
label_long = (
    future_20d_ret_wide
    .stack(future_stack=True)
    .rename("future_20d_ret")
    .reset_index()
    .rename(columns={"level_0": "date", "level_1": "ticker"})
)

label_long.head(10)


### 9.5 加上当天收益和价格


In [ ]:
daily_ret_long = (
    daily_ret_wide
    .stack(future_stack=True)
    .rename("daily_ret")
    .reset_index()
    .rename(columns={"level_0": "date", "level_1": "ticker"})
)

price_long = (
    price_wide
    .stack(future_stack=True)
    .rename("adj_close")
    .reset_index()
    .rename(columns={"level_0": "date", "level_1": "ticker"})
)

research_base = (
    price_long
    .merge(daily_ret_long, on=["date", "ticker"], how="left")
    .merge(label_long, on=["date", "ticker"], how="left")
)

research_base.head(10)


这个表已经很像因子研究的底座了。

---

## 10. 构造因子值，并和标签对齐

为了说明对齐，我们构造一个简单的 20 日动量因子：


momentum_20d(t) = price(t) / price(t-20) - 1


它使用的是过去 20 日收益，所以在 t 日可以知道。

### 10.1 构造过去 20 日动量因子


In [ ]:
momentum_20d_wide = price_wide / price_wide.shift(20) - 1

factor_long = (
    momentum_20d_wide
    .stack(future_stack=True)
    .rename("momentum_20d")
    .reset_index()
    .rename(columns={"level_0": "date", "level_1": "ticker"})
)

factor_long.head(10)


### 10.2 合并因子和未来收益标签


In [ ]:
dataset = (
    factor_long
    .merge(label_long, on=["date", "ticker"], how="inner")
    .dropna(subset=["momentum_20d", "future_20d_ret"])
)

dataset.head()


现在每一行的含义是：


在 date 这一天，
ticker 这只股票过去 20 日动量是 momentum_20d，
之后 20 个交易日收益是 future_20d_ret。


这就是后面计算 IC 的基础表。

### 10.3 做一个极简相关性检查

第 3 天会正式学 IC。今天先粗略感受一下：


In [ ]:
simple_corr = dataset["momentum_20d"].corr(dataset["future_20d_ret"])
print("momentum_20d 与 future_20d_ret 的整体相关性:", round(simple_corr, 4))


如果你看到相关性很小，不要惊讶。

真实市场里，单个简单因子的信号通常很弱。因子研究不是寻找“百分百预测”，而是寻找在大样本中略微稳定的倾向。

---

## 11. 一个非常关键的时间点问题

到这里，我们已经能构造未来 20 日收益标签。

但实盘里还有一个问题：

> 你的因子值是在什么时候知道的？你能在什么价格成交？

### 11.1 三种常见标签定义

| 标签 | 公式 | 适用直觉 |
| --- | --- | --- |
| close-to-close | `close[t+h] / close[t] - 1` | 假设 t 日收盘即可形成并执行信号 |
| next-close-to-close | `close[t+h+1] / close[t+1] - 1` | t 日收盘后生成信号，下一交易日收盘附近成交 |
| next-open-to-close | `close[t+h] / open[t+1] - 1` | t 日收盘后生成信号，下一交易日开盘成交 |

初学阶段用 close-to-close 可以理解框架，但实盘研究要更严格。

如果因子值依赖 t 日收盘价，比如动量、波动率、换手率，那你通常不能假设自己已经在 t 日收盘价成交。

更保守的做法是：


t 日收盘后得到因子
t+1 日买入
t+1+h 日或 t+h 日附近卖出


### 11.2 用 next-close-to-close 标签

我们构造一个更保守的标签：


In [ ]:
def forward_return_next_close(price: pd.DataFrame, holding_days: int) -> pd.DataFrame:
    """
    假设 t 日收盘后形成信号，t+1 日收盘买入，
    持有 holding_days 个交易日后卖出。

    entry = price[t+1]
    exit = price[t+1+holding_days]
    """
    entry = price.shift(-1)
    exit_ = price.shift(-(1 + holding_days))
    return exit_ / entry - 1


future_20d_next_close_wide = forward_return_next_close(price_wide, 20)

comparison = pd.DataFrame({
    "close_to_close": future_20d_ret_wide["AAA"],
    "next_close_to_close": future_20d_next_close_wide["AAA"],
})

comparison.head(8)


你会看到两个标签很像，但不完全一样。

这种差别会影响回测结果。

### 11.3 什么时候用哪一种？

如果你只是学习概念：


close[t+20] / close[t] - 1


足够清楚。

如果你做正式研究：


先定义信号生成时间，再定义可交易价格，再定义卖出时间。


没有这个步骤，回测很容易悄悄变乐观。

---

## 12. 缺失值、停牌和样本边界

真实数据不会像模拟数据这么整齐。

常见问题：

- 股票新上市，前面没有足够历史价格。
- 股票停牌，某些交易日没有价格。
- 股票退市，后面没有未来价格。
- 涨跌停导致无法按假设价格成交。
- 价格是 0、负数或异常跳变。

今天先处理最基础的缺失值。

### 12.1 人为制造缺失


In [ ]:
price_with_nan = price_wide.copy()
price_with_nan.loc[price_with_nan.index[30:35], "CCC"] = np.nan
price_with_nan.loc[price_with_nan.index[80:83], "EEE"] = np.nan

future_20d_with_nan = price_with_nan.shift(-20) / price_with_nan - 1

nan_check = pd.DataFrame({
    "price_nan_count": price_with_nan.isna().sum(),
    "label_nan_count": future_20d_with_nan.isna().sum(),
})

nan_check


缺失价格会传导到标签。

这不是坏事。它提醒你：这部分样本不能可靠计算。

### 12.2 不要随便填充未来标签

危险做法：


future_20d_ret.fillna(0)


为什么危险？

因为缺失标签不是“收益为 0”，而是“不知道未来收益”。

正确做法通常是：


In [ ]:
clean_label_long = (
    future_20d_with_nan
    .stack(future_stack=True)
    .dropna()
    .rename("future_20d_ret")
    .reset_index()
    .rename(columns={"level_0": "date", "level_1": "ticker"})
)

clean_label_long.head()


对于标签缺失，初学阶段最稳妥的处理是丢弃。

对于价格缺失是否前向填充，要看数据源含义：

- 如果是停牌，前向填充可能低估风险和交易限制。
- 如果是非交易日对齐问题，前向填充可能合理。
- 如果是数据缺漏，应该回查数据源。

---

## 13. 目标产出：未来 20 日收益标签函数

现在把今天的核心内容封装成一个函数。

这个函数输入宽表价格矩阵：


index = date
columns = ticker
values = adjusted close price


输出长表标签：


date | ticker | future_20d_ret


### 13.1 close-to-close 版本


In [ ]:
def make_forward_return_label(
    adj_close: pd.DataFrame,
    holding_days: int = 20,
    label_name: str | None = None,
) -> pd.DataFrame:
    """
    构建未来 N 日收益标签。

    参数
    ----
    adj_close:
        复权收盘价宽表，index 为交易日，columns 为股票代码。
    holding_days:
        持有期，单位为交易日。
    label_name:
        标签列名。默认 future_{holding_days}d_ret。

    返回
    ----
    DataFrame:
        长表，包含 date、ticker、标签列。
    """
    if holding_days <= 0:
        raise ValueError("holding_days 必须是正整数")

    if label_name is None:
        label_name = f"future_{holding_days}d_ret"

    forward_ret = adj_close.shift(-holding_days) / adj_close - 1

    label = (
        forward_ret
        .stack(future_stack=True)
        .dropna()
        .rename(label_name)
        .reset_index()
        .rename(columns={"level_0": "date", "level_1": "ticker"})
    )

    return label


future_20d_label = make_forward_return_label(price_wide, holding_days=20)
future_20d_label.head()


### 13.2 验证函数结果


In [ ]:
sample_row = future_20d_label.iloc[7]
sample_date = sample_row["date"]
sample_ticker = sample_row["ticker"]

date_pos = price_wide.index.get_loc(sample_date)
exit_date = price_wide.index[date_pos + 20]

manual = price_wide.loc[exit_date, sample_ticker] / price_wide.loc[sample_date, sample_ticker] - 1
from_function = sample_row["future_20d_ret"]

print("样本日期:", sample_date.date())
print("股票:", sample_ticker)
print("退出日期:", exit_date.date())
print("手工计算:", f"{manual:.6%}")
print("函数结果:", f"{from_function:.6%}")
print("是否一致:", np.isclose(manual, from_function))


### 13.3 批量生成不同持有期标签


In [ ]:
label_list = []

for h in [1, 5, 10, 20, 60]:
    one_label = make_forward_return_label(price_wide, holding_days=h)
    label_list.append(one_label)

multi_horizon_labels = label_list[0]

for one_label in label_list[1:]:
    multi_horizon_labels = multi_horizon_labels.merge(
        one_label,
        on=["date", "ticker"],
        how="outer"
    )

multi_horizon_labels.head()


这张表就是“不同持有期收益”的基础成果。

---

## 14. 标签质量检查清单

每次构建收益标签，都建议检查下面 8 件事。

### 14.1 是否使用复权价格？

如果使用未复权价格，先确认你的研究目的真的需要未复权价格。

### 14.2 `shift` 方向是否正确？

未来收益一般用：


price.shift(-holding_days) / price - 1


过去收益一般用：


price / price.shift(holding_days) - 1


### 14.3 标签是否和因子日期对齐？

同一行应该表示：


date 当天能看到的因子值
date 之后发生的未来收益


### 14.4 最后 N 天是否自然缺失？

未来 20 日标签最后 20 个交易日没有未来价格，应该缺失。

### 14.5 第一批样本是否因历史窗口缺失？

如果你的因子需要过去 20 日价格，那么前 20 日因子值应该缺失。

### 14.6 是否误把缺失标签填 0？

不要这么做。

缺失是“不知道”，不是“没有涨跌”。

### 14.7 持有期是否和研究问题一致？

不要随便拿 20 日作为万能答案。

### 14.8 是否抽样手工验证？

至少抽 3 行：

- 第一段样本
- 中间样本
- 接近末尾的样本

手工算一遍。

---

## 15. 一个完整小项目：构建因子研究底表

今天最后做一个完整小项目。

目标：


构造一张包含：
date, ticker, adj_close, daily_ret, momentum_20d, future_20d_ret
的研究底表。


### 15.1 写一个底表函数


In [ ]:
def wide_to_long(wide: pd.DataFrame, value_name: str) -> pd.DataFrame:
    return (
        wide
        .stack(future_stack=True)
        .dropna()
        .rename(value_name)
        .reset_index()
        .rename(columns={"level_0": "date", "level_1": "ticker"})
    )


def build_research_table(adj_close: pd.DataFrame, holding_days: int = 20) -> pd.DataFrame:
    """
    构建一个最小可用的因子研究底表。
    """
    price_long = wide_to_long(adj_close, "adj_close")
    daily_ret_long = wide_to_long(adj_close.pct_change(), "daily_ret")
    momentum_long = wide_to_long(adj_close / adj_close.shift(20) - 1, "momentum_20d")
    label_long = make_forward_return_label(adj_close, holding_days=holding_days)

    table = (
        price_long
        .merge(daily_ret_long, on=["date", "ticker"], how="left")
        .merge(momentum_long, on=["date", "ticker"], how="left")
        .merge(label_long, on=["date", "ticker"], how="left")
        .dropna(subset=["momentum_20d", f"future_{holding_days}d_ret"])
        .sort_values(["date", "ticker"])
        .reset_index(drop=True)
    )

    return table


research_table = build_research_table(price_wide, holding_days=20)
research_table.head(10)


### 15.2 检查底表


In [ ]:
checks = {
    "rows": len(research_table),
    "unique_dates": research_table["date"].nunique(),
    "unique_tickers": research_table["ticker"].nunique(),
    "missing_values": int(research_table.isna().sum().sum()),
    "min_date": research_table["date"].min().date(),
    "max_date": research_table["date"].max().date(),
}

checks


### 15.3 按日期看每期股票数量


In [ ]:
count_by_date = research_table.groupby("date")["ticker"].nunique()

count_by_date.plot(title="每天可用股票数量")
plt.ylabel("Stock Count")
plt.show()


如果是真实数据，这张图很有用。

它能让你发现：

- 某些日期样本突然变少。
- 某些股票池成分变化异常。
- 标签和因子窗口导致的头尾缺失是否符合预期。

---

## 16. 今天的知识图谱


In [ ]:
mindmap
  root((收益率与标签))
    收益率
      简单收益率
        price_today / price_yesterday - 1
        直观
        适合组合收益
      对数收益率
        log(price_today / price_yesterday)
        多期可相加
        适合统计建模
    价格
      原始价格
        会受除权除息影响
      复权价格
        更适合收益计算
      异常价格
        需要检查
    持有期
      1日
      5日
      20日
      60日
      不同周期对应不同问题
    标签
      future_1d_ret
      future_5d_ret
      future_20d_ret
      future_60d_ret
    时间对齐
      因子值在t日可知
      标签来自t日之后
      shift负数取未来
      shift正数取过去
    风险点
      未来函数
      使用未复权价格
      标签填0
      持有期错位
      停牌和缺失
    目标产出
      价格宽表
      标签长表
      研究底表


文本版：


收益率与标签
├── 收益率
│   ├── 简单收益率
│   └── 对数收益率
├── 价格选择
│   ├── 原始价格
│   └── 复权价格
├── 持有期
│   ├── 1日
│   ├── 5日
│   ├── 20日
│   └── 60日
├── 标签构建
│   ├── future_1d_ret
│   ├── future_20d_ret
│   └── 多持有期标签
├── 时间对齐
│   ├── t日因子
│   ├── t+h日收益
│   └── 避免未来函数
└── 质量检查
    ├── shift方向
    ├── 头尾缺失
    ├── 复权检查
    └── 手工抽样验证


---

## 17. 初学者最容易踩的 9 个坑

### 坑 1：把过去收益当未来标签


price / price.shift(20) - 1


这是过去 20 日收益，不是未来 20 日收益。

### 坑 2：把未来标签拿来做因子

如果你把 `future_20d_ret` 当作特征放进模型，模型会表现得像天才，但那是偷看答案。

### 坑 3：使用未复权价格计算长期收益

分红、拆股、送股会污染收益率。

### 坑 4：忘记最后 N 天标签缺失

未来 20 日收益的最后 20 个交易日没有未来数据，缺失是正常的。

### 坑 5：把缺失标签填成 0

缺失不是收益为 0。

### 坑 6：忽略信号生成时间

如果你的因子在收盘后才能算出来，却假设自己按当天收盘价买入，回测会偏乐观。

### 坑 7：混淆交易日和自然日

20 个交易日大约是一个月，不是 20 个自然日。

### 坑 8：没处理股票池变化

新上市、退市、停牌都会影响样本。

### 坑 9：只看一个持有期

因子可能在 5 日有效、20 日一般、60 日失效。多持有期检查能帮你理解因子衰减。

---

## 18. 今天的动手作业

### 作业 A：解释收益率

用自己的话回答：

1. 为什么不能直接用价格涨跌金额比较不同股票？
2. 简单收益率和对数收益率有什么区别？
3. 为什么做长期收益标签时要注意复权价格？

### 作业 B：手写未来 20 日收益公式

写出：


future_20d_ret(t) =


并说明：

- 分子是哪一天的价格？
- 分母是哪一天的价格？
- 这个标签在最后 20 个交易日为什么为空？

### 作业 C：运行今天的 Python 代码

运行本文所有 Python 代码，并确认你能得到：

1. 一只股票的日收益率。
2. 一只股票的未来 20 日收益标签。
3. 多股票的未来 20 日收益标签长表。
4. 包含 `momentum_20d` 和 `future_20d_ret` 的研究底表。

### 作业 D：改造持有期

把持有期从 20 改成：

- 5
- 10
- 60

观察：

- 样本数量如何变化？
- 标签波动如何变化？
- 和 `momentum_20d` 的相关性是否变化？

### 作业 E：写一个自己的标签函数

要求：


In [ ]:
def make_label(adj_close, holding_days):
    ...


输入：

- `adj_close`：复权收盘价宽表。
- `holding_days`：持有期。

输出：

- 长表：`date, ticker, future_{holding_days}d_ret`

加分项：

- 检查 `holding_days` 必须大于 0。
- 自动删除缺失标签。
- 抽样验证一行。

---

## 19. 自测题

### 题 1

`price.pct_change()` 算出来的是过去收益还是未来收益？

答案：过去收益。它表示今天价格相对上一期价格的变化。

### 题 2

未来 20 日收益标签应该用 `shift(20)` 还是 `shift(-20)`？

答案：通常用 `shift(-20)`，因为要把未来第 20 个交易日的价格搬到今天这一行。

### 题 3

为什么未来 20 日收益最后 20 行是缺失？

答案：因为这些日期后面没有足够的 20 个交易日价格，无法知道未来 20 日收益。

### 题 4

缺失的未来收益标签能不能填 0？

答案：不能。缺失代表无法计算，不代表收益为 0。

### 题 5

如果一个因子使用 t 日收盘价才能计算出来，是否一定能用 t 日收盘价成交？

答案：不一定。更保守的研究通常假设 t 日收盘后形成信号，t+1 日成交。

### 题 6

做收益率标签时，为什么优先使用复权价格？

答案：因为分红、拆股、送股等公司行为会影响原始价格，复权价格更能反映真实持有收益。

---

## 20. 今日复盘模板


第 02 天复盘：收益率与标签

1. 我今天理解的简单收益率：

2. 我今天理解的对数收益率：

3. 我今天理解的未来 20 日收益标签：

4. 我最容易写错的 shift 方向：

5. 我对复权价格的理解：

6. 我构建出的标签表字段：

7. 我今天手工验证的一行标签：

8. 明天学习 IC 前，我还需要补的知识：


---

## 21. 明天预告：IC 基础

明天会学习 IC。

IC 的问题是：

> 在同一个日期里，因子值高的股票，未来收益是否也更高？

也就是说，今天构造的这张表：


date | ticker | factor_value | future_20d_ret


会在明天变成 IC 计算的输入。

你今天把标签做对，明天的 IC 才有意义。

---

## 22. 一句话收尾

收益率与标签看起来像数据清洗，其实是因子研究的地基。

> 标签一错，后面的 IC、回测、机器学习模型都会认真地回答一个错误问题。

今天最重要的习惯是：每写一个标签公式，就问自己三遍：


这个价格在当时能知道吗？
这个收益真的是未来吗？
这个日期和因子值对齐了吗？


---

## 23. 仅供学习的提醒

本文所有示例使用模拟数据，只用于解释收益率和标签构建方法，不构成任何投资建议。真实研究需要使用可靠的数据源、准确的复权价格、严格的交易时点假设，以及对停牌、退市、涨跌停、交易成本等问题的处理。

---

# 统一高质量增强模块

> 本增强模块用于把第 02 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：收益率与标签
- 必做：收益率计算
- 选做：不同持有期收益
- 目标产出：构建未来20日收益标签

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

因子像学生交卷，未来收益像标准答案。标准答案日期错了，后面 IC、回测、机器学习都会认真地批改一份错误试卷。

这个例子背后的关键直觉是：

> 标签是未来答案，但必须和当时可得的因子严格对齐。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


收益率与标签
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 构建未来20日收益标签


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(102)
dates = pd.bdate_range("2024-01-02", periods=120)
close = pd.Series(100 * np.cumprod(1 + rng.normal(0.0005, 0.015, len(dates))), index=dates, name="close")

def make_forward_return(price: pd.Series, holding_days: int) -> pd.Series:
    return price.shift(-holding_days) / price - 1

table = pd.DataFrame({"close": close})
for h in [1, 5, 20, 60]:
    table[f"future_{h}d_ret"] = make_forward_return(close, h)

check_date = table.index[10]
manual = table.loc[table.index[30], "close"] / table.loc[check_date, "close"] - 1
auto = table.loc[check_date, "future_20d_ret"]
print(table.head())
print("manual equals auto:", np.isclose(manual, auto))


## E. 产出验收标准

完成今天课程后，你的 `构建未来20日收益标签` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `收益率与标签` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `构建未来20日收益标签` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `构建未来20日收益标签`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 02 天复盘：收益率与标签

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
